In [0]:
!pip install seaborn

Please use this notebook to perform model testing. Perform the following tasks:
- Import silver dataset and split appropriately. Record splitting strategy in dictionary.
- Build model. Record selected model and hyperparams in dictionary.
- Score model on validation and test set. Record all scores in dictionary.
    - For regression, MAE, RMSE, MAPE and R2
    - For classification, ROC AUC, Precision, Recall, Accuracy, Conf mat
- Bin all numerical features over 10 sets, record residuals over all categories. Record in dictionary.


In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import os 
import scipy
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder.getOrCreate()
df_gold = spark.sql("""
                    select * from workspace.ds_cb_wind_turbine.silver
                    """).toPandas()

In [0]:
from datetime import datetime

model_scoring_report = {}

train_month_threshold = datetime(2018, 3, 1, 0, 0, 0, 0)
val_month_threshold = datetime(2018, 4, 1, 0, 0, 0, 0)
test_month_threshold = datetime(2018, 8, 1, 0, 0, 0, 0)

model_scoring_report['splitting_strategy'] = {
    'name': 'time series split',
    'train_threshold': f'{str(train_month_threshold)}',
    'val_threshold': f'{str(val_month_threshold)}',
    'test_threshold': f'{str(test_month_threshold)}'
    }

feature_cols = ['wind_speed', 'wind_direction_rad', 'sine_hour', 'cosine_hour', 'wind_magnitude_i', 'wind_magnitude_j', 'wind_magnitude_i_3h_rolling', 'wind_magnitude_j_3h_rolling', 'wind_speed_3h_rolling']
target_col = ['lv_active_power']

df_train = df_gold.loc[df_gold['timestamp'] < train_month_threshold]
df_val = df_gold.loc[(df_gold['timestamp'] >= train_month_threshold) & (df_gold['timestamp'] <= val_month_threshold)]

In [0]:
selected_feature_description_dict = {
    'wind_speed': 'Wind speed in m/s',
    'wind_direction_rad': 'Wind direction in radians',
    'sine_hour': 'Sine of the hour of the day',
    'cosine_hour': 'Cosine of the hour of the day',
    'wind_magnitude_i': 'Wind magnitude in the i direction',
    'wind_magnitude_j': 'Wind magnitude in the j direction',
    'wind_magnitude_i_3h_rolling': 'Wind magnitude in the i direction, 3h rolling average',
    'wind_magnitude_j_3h_rolling': 'Wind magnitude in the j direction, 3h rolling average',
    'wind_speed_3h_rolling': 'Wind speed 3h rolling average'
}

model_scoring_report['features'] = selected_feature_description_dict

In [0]:
X_train = df_train[feature_cols]
y_train = df_train[target_col]

X_val = df_val[feature_cols]
y_val = df_val[target_col]
t_val = df_val['timestamp']

print(len(X_train))
print(len(X_val))

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

rf_params = {
    "n_estimators": 300,        # enough trees to stabilize predictions without excessive runtime
    "max_depth": 15,            # caps tree complexity; prevents overfitting on ~50k rows with few features
    "min_samples_split": 10,    # requires a reasonable sample size before splitting further
    "min_samples_leaf": 5,      # avoids leaves fit to tiny, noisy subsets
    "max_features": "sqrt",     # standard default for RF; decorrelates trees
    "n_jobs": -1,                # use all available cores
    "random_state": 42,          # reproducibility
}

rf = RandomForestRegressor(**rf_params)

model_scoring_report['model'] = 'RandomForestRegressor'
model_scoring_report['hyperparams'] = rf_params


rf.fit(X_train, y_train)

test_y_hat = rf.predict(X_val)

predictions_set = X_val.copy()

predictions_set['y'] = y_val
predictions_set['y_hat'] = test_y_hat

predictions_set

In [0]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

first_val_mae = mean_absolute_error(y_val, test_y_hat)
print(first_val_mae)

In [0]:
first_val_rmse = root_mean_squared_error(y_val, test_y_hat)

print(first_val_rmse)

In [0]:
first_val_r2 = r2_score(y_val, test_y_hat)

print(first_val_r2)

In [0]:
scoring_df = predictions_set.copy()
scoring_df['residual'] = scoring_df['y'] - scoring_df['y_hat']

residual_distribution = {}
for col in feature_cols:
    scoring_df[f'{col}_bin_10'] = pd.cut(scoring_df[col], 10).astype(str)
    residuals_means = scoring_df.groupby(f'{col}_bin_10')['residual'].mean().reset_index().to_dict()
    residuals_stds = scoring_df.groupby(f'{col}_bin_10')['residual'].std().reset_index().to_dict()
    residual_distribution[col] = {
        'mean': residuals_means,
        'std': residuals_stds
    }

model_scoring_report['residuals'] = residual_distribution


In [0]:
model_scoring_report['metrics'] = {
    'mae': first_val_mae,
    'rmse': first_val_rmse,
    'r2': first_val_r2
}

In [0]:
print(model_scoring_report)

In [0]:
import json

In [0]:

model_scoring_json = json.dumps(model_scoring_report, indent=2)

with open('/Workspace/Users/corey.bevan@gmail.com/wind_turbine_drift_detection/analysis/gold/model_report.json', 'w') as f:
    json.dump(model_scoring_report, f, indent=2)

print(model_scoring_json)